### **<h3 style="color:pink;"> RAG System — Week 6: Fine-tuning Data Preparation**

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Introduction**</span>

</div>

This week we prepare the training dataset for fine-tuning Mistral-7B!

**What we'll do:**
- ✅ Generate 2000+ instruction-answer pairs from our legal documents
- ✅ Format data in Alpaca format (what Mistral-7B expects)
- ✅ Split into train/validation/test sets
- ✅ Save dataset ready for Google Colab fine-tuning!

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Setup & Imports**</span>

</div>

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import os
import random
import time
from langchain_groq import ChatGroq

random.seed(42)

print("✅ All imports successful!")

✅ All imports successful!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Setup Groq & Load Data**</span>

</div>

In [ ]:
# Setup Groq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,  # slightly creative for diverse examples
    api_key="GROQ_API_KEY"  # paste your key here
)

# Load documents
with open("../data/raw/legal_documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

# Load our 200 QA pairs from Week 2
with open("../data/processed/qa_pairs.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

print(f"✅ Groq connected!")
print(f"✅ Loaded {len(documents)} documents")
print(f"✅ Loaded {len(qa_pairs)} existing QA pairs")

✅ Groq connected!
✅ Loaded 500 documents
✅ Loaded 200 existing QA pairs


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Understanding the Training Format**</span>

</div>

Mistral-7B expects data in **Alpaca format**:
- instruction = what the model should do
- input = the context + question
- output = the correct answer

In [4]:
# Show exactly what format we need
example = {
    "instruction": "You are a legal expert. Answer the following question based ONLY on the provided legal document context. Be precise and faithful to the source.",
    "input": """Context: SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS. A business entity shall not be subject to civil liability relating to any injury or death occurring at a facility if the use occurs outside the scope of business.

Question: Under what condition is a business entity not liable for injuries?""",
    "output": "A business entity is not liable for injuries when the use of its facility occurs outside the scope of its normal business operations."
}

print("📋 Example training sample:")
print(json.dumps(example, indent=2))
print(f"\n✅ This is the Alpaca format Mistral-7B expects!")
print(f"\n🎯 We need 2000+ examples like this!")

📋 Example training sample:
{
  "instruction": "You are a legal expert. Answer the following question based ONLY on the provided legal document context. Be precise and faithful to the source.",
  "input": "Context: SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS. A business entity shall not be subject to civil liability relating to any injury or death occurring at a facility if the use occurs outside the scope of business.\n\nQuestion: Under what condition is a business entity not liable for injuries?",
  "output": "A business entity is not liable for injuries when the use of its facility occurs outside the scope of its normal business operations."
}

✅ This is the Alpaca format Mistral-7B expects!

🎯 We need 2000+ examples like this!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Building Training Data Generator**</span>

</div>

In [5]:
INSTRUCTION = "You are a legal expert. Answer the following question based ONLY on the provided legal document context. Be precise and faithful to the source."

def generate_training_example(chunk_text):
    prompt = f"""You are a legal expert. Given this legal document excerpt, generate ONE question and its precise answer.

Document excerpt:
{chunk_text}

Rules:
- Question must be answerable ONLY from the excerpt
- Answer must be precise (1-3 sentences)
- Focus on factual legal details
- Answer must be faithful to the document

Respond in this exact JSON format:
{{"question": "your question here", "answer": "your answer here"}}

JSON only, no other text."""

    response = llm.invoke(prompt).content.strip()
    
    # Clean and parse JSON
    if response.startswith("```"):
        response = response.split("```")[1]
        if response.startswith("json"):
            response = response[4:]
    
    parsed = json.loads(response)
    
    # Format as Alpaca training example
    return {
        "instruction": INSTRUCTION,
        "input": f"Context: {chunk_text}\n\nQuestion: {parsed['question']}",
        "output": parsed["answer"]
    }

# Test it
print("🔍 Testing generator...")
test_chunk = documents[0]["text"][:400]
example = generate_training_example(test_chunk)

print(f"✅ Generated example:")
print(f"   Q: {example['input'].split('Question: ')[1]}")
print(f"   A: {example['output']}")

🔍 Testing generator...
✅ Generated example:
   Q: What is defined as a 'business entity' in this section?
   A: A business entity is defined as a firm, corporation, association, partnership, consortium, joint venture, or other form of enterprise.


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Generating 2000+ Training Examples**</span>

</div>

⏳ This will take ~3-4 hours due to Groq rate limits.
We'll generate examples in batches and save progress automatically!

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create chunks from documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks = []
for doc in documents:
    doc_chunks = splitter.split_text(doc["text"])
    for chunk in doc_chunks:
        if len(chunk) > 100:  # skip very short chunks
            all_chunks.append(chunk)

print(f"✅ Created {len(all_chunks)} chunks for training")

# Randomly select 2000 chunks
selected_chunks = random.sample(all_chunks, min(1000, len(all_chunks)))
print(f"✅ Selected {len(selected_chunks)} chunks to generate examples from")

✅ Created 14121 chunks for training
✅ Selected 1000 chunks to generate examples from


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Generating 1000 Training Examples**</span>

</div>

⏳ ~90 minutes — progress saved every 50 examples automatically!

In [8]:
import os

# Output path
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/training_data.json"

# Load existing progress if any
if os.path.exists(output_path):
    with open(output_path, "r", encoding="utf-8") as f:
        training_data = json.load(f)
    print(f"📂 Resuming from {len(training_data)} existing examples!")
else:
    training_data = []
    print(f"🆕 Starting fresh!")

# Skip already processed chunks
start_idx = len(training_data)
remaining_chunks = selected_chunks[start_idx:]

print(f"⏳ Generating {len(remaining_chunks)} examples...")
print(f"💾 Progress saved every 50 examples")

failed = 0
start_time = time.time()

for i, chunk in enumerate(remaining_chunks):
    try:
        example = generate_training_example(chunk)
        training_data.append(example)

        # Save every 50 examples
        if (i + 1) % 50 == 0:
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(training_data, f, ensure_ascii=False, indent=2)
            elapsed = time.time() - start_time
            total_done = start_idx + i + 1
            print(f"💾 Saved! {total_done}/1000 done ({elapsed/60:.1f} mins elapsed)")

        time.sleep(0.5)  # avoid rate limiting

    except Exception as e:
        failed += 1
        if failed <= 5:
            print(f"⚠️ Failed example {i}: {e}")
        time.sleep(2)

# Final save
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(training_data, f, ensure_ascii=False, indent=2)

elapsed = time.time() - start_time
print(f"\n✅ Done! Generated {len(training_data)} examples in {elapsed/60:.1f} minutes")
print(f"⚠️ Failed: {failed}")
print(f"💾 Saved to {output_path}")

🆕 Starting fresh!
⏳ Generating 1000 examples...
💾 Progress saved every 50 examples
💾 Saved! 50/1000 done (1.6 mins elapsed)
💾 Saved! 100/1000 done (4.2 mins elapsed)
💾 Saved! 150/1000 done (6.7 mins elapsed)
💾 Saved! 200/1000 done (9.3 mins elapsed)
💾 Saved! 250/1000 done (11.8 mins elapsed)
💾 Saved! 300/1000 done (14.3 mins elapsed)
⚠️ Failed example 325: Expecting ',' delimiter: line 1 column 289 (char 288)
💾 Saved! 350/1000 done (16.8 mins elapsed)
💾 Saved! 400/1000 done (19.4 mins elapsed)
💾 Saved! 450/1000 done (21.9 mins elapsed)
💾 Saved! 500/1000 done (24.4 mins elapsed)
💾 Saved! 550/1000 done (26.9 mins elapsed)
💾 Saved! 600/1000 done (29.5 mins elapsed)
💾 Saved! 650/1000 done (31.9 mins elapsed)
💾 Saved! 700/1000 done (34.3 mins elapsed)
💾 Saved! 750/1000 done (36.8 mins elapsed)
💾 Saved! 800/1000 done (39.2 mins elapsed)
💾 Saved! 850/1000 done (41.6 mins elapsed)
💾 Saved! 900/1000 done (44.1 mins elapsed)
💾 Saved! 950/1000 done (46.5 mins elapsed)
💾 Saved! 1000/1000 done (48.

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Splitting Dataset — Train/Validation/Test**</span>

</div>

In [9]:
# Load generated training data
with open("../data/processed/training_data.json", "r", encoding="utf-8") as f:
    training_data = json.load(f)

print(f"✅ Loaded {len(training_data)} training examples")

# Shuffle data
random.shuffle(training_data)

# Split 80% train, 10% validation, 10% test
total = len(training_data)
train_end = int(total * 0.8)
val_end = int(total * 0.9)

train_data = training_data[:train_end]
val_data   = training_data[train_end:val_end]
test_data  = training_data[val_end:]

print(f"\n📊 Dataset Split:")
print(f"   Train      : {len(train_data)} examples (80%)")
print(f"   Validation : {len(val_data)} examples (10%)")
print(f"   Test       : {len(test_data)} examples (10%)")

# Save splits
with open("../data/processed/train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open("../data/processed/validation.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

with open("../data/processed/test.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ All splits saved to disk!")

✅ Loaded 999 training examples

📊 Dataset Split:
   Train      : 799 examples (80%)
   Validation : 100 examples (10%)
   Test       : 100 examples (10%)

✅ All splits saved to disk!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Week 6 Summary**</span>

</div>

In [10]:
import os

print("=" * 60)
print("🎉 WEEK 6 - FINE-TUNING DATA PREPARATION COMPLETE!")
print("=" * 60)

print("""
📦 What we built today:
   ✅ Generated 999 legal instruction-answer pairs
   ✅ Formatted in Alpaca format for Mistral-7B
   ✅ Split into train/validation/test sets
   ✅ Saved everything to disk
""")

print("📁 Files saved:")
for path in [
    "../data/processed/training_data.json",
    "../data/processed/train.json",
    "../data/processed/validation.json",
    "../data/processed/test.json"
]:
    size = os.path.getsize(path) / 1024
    name = path.split("/")[-1]
    print(f"   📄 {name:<25} {size:.1f} KB")

print(f"""
📊 Dataset Summary:
   Total examples : 999
   Train          : 799 (80%)
   Validation     : 100 (10%)
   Test           : 100 (10%)
   Format         : Alpaca (instruction/input/output)

🔜 Next — Week 7: QLoRA Fine-tuning on Google Colab
   → Upload train.json + validation.json to Colab
   → Fine-tune Mistral-7B with QLoRA
   → Expected faithfulness: 0.840+
   → This is the BIG improvement! 🚀
""")
print("=" * 60)

🎉 WEEK 6 - FINE-TUNING DATA PREPARATION COMPLETE!

📦 What we built today:
   ✅ Generated 999 legal instruction-answer pairs
   ✅ Formatted in Alpaca format for Mistral-7B
   ✅ Split into train/validation/test sets
   ✅ Saved everything to disk

📁 Files saved:
   📄 training_data.json        733.8 KB
   📄 train.json                585.5 KB
   📄 validation.json           73.6 KB
   📄 test.json                 74.7 KB

📊 Dataset Summary:
   Total examples : 999
   Train          : 799 (80%)
   Validation     : 100 (10%)
   Test           : 100 (10%)
   Format         : Alpaca (instruction/input/output)

🔜 Next — Week 7: QLoRA Fine-tuning on Google Colab
   → Upload train.json + validation.json to Colab
   → Fine-tune Mistral-7B with QLoRA
   → Expected faithfulness: 0.840+
   → This is the BIG improvement! 🚀

